# Cascaded inference and comparison

## Goal
Run a portable bundle in separate backend workers while preserving input IDs.

## Setup
Install the package in this kernel's environment. All default data are synthetic; research training is disabled.

In [ ]:
from pathlib import Path
import os, sys, uuid
import numpy as np
PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), Path.cwd())
WORK = Path(os.environ.get("SERS_NOTEBOOK_OUTPUT", PROJECT / "output/notebooks")) / uuid.uuid4().hex[:8]
WORK.mkdir(parents=True)
from sersd2d_superraman.data import SpectrumBatch, load_batch, save_batch
from sersd2d_superraman.cli import synthetic, prepare

### 1. Select artifacts

In [ ]:
from sersd2d_superraman.pipeline import run_pipeline
BUNDLE = Path(os.environ.get("SERS_NOTEBOOK_BUNDLE", PROJECT / "output/synthetic_demo/bundle/bundle.json"))
INPUT = Path(os.environ.get("SERS_NOTEBOOK_INPUT", PROJECT / "output/synthetic_demo/prepared/Enwave/test.npz"))
print("Create these synthetic artifacts first with: sers-pipeline demo --output output/synthetic_demo")

### 2. Execute the cascade

In [ ]:
report = None
if BUNDLE.is_file() and INPUT.is_file():
    report = run_pipeline(BUNDLE, INPUT, WORK / "prediction", batch_size=4)
    print({k: report[k] for k in ("source_device", "rows", "contains_untrained_models")})
else:
    print("Bundle/input unavailable. Run the demo, or select your verified research artifacts, then rerun.")

## Checks

In [ ]:
if report is not None:
    assert (WORK / "prediction/translated.npz").is_file()
    with np.load(WORK / "prediction/original_predictions.npz") as before, np.load(WORK / "prediction/translated_predictions.npz") as after:
        assert np.array_equal(before["sample_ids"], after["sample_ids"])
        assert before["logits"].shape == after["logits"].shape

## Next steps
Synthetic comparison scores do not establish efficacy. For research, use an independently evaluated bundle and record data/model hashes. Translation metrics require an independent Renishaw reference batch.